# Week 1 analysis - classic DNS vs DoH latency

**Purpose:** first look at your own measurement data. If something is fundamentally wrong (empty columns, every query failing, DoH impossibly fast) you find out now, not in week 4.

Reads every `results/raw_*.csv` (both teammates' files, after `git pull`).

In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# find the repo root (the folder that contains client/) no matter where Jupyter was started
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "client").is_dir())
for sub in ("client", "ml", "capture"):
    sys.path.insert(0, str(ROOT / sub))
RESULTS = ROOT / "results"
print("repo root:", ROOT)

repo root: /home/sumit/cn-project/doh-measurement-project


In [2]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from summarize import load_raw, summarize, summarize_by_run
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

df = load_raw(RESULTS)
print(len(df), "rows from", df.source_file.nunique(), "files;  people:", sorted(df.person.unique()),
      " profiles:", sorted(df.profile.unique()))
df.head()

2727 rows from 9 files;  people: ['A', 'B']  profiles: ['clean']


,person,profile,run_id,seq,domain,resolver,method,qtype,server,send_ts,recv_ts,rtt_ms,status,rcode,answer_count,resolved_ips,attempts,error,source_file
0,A,clean,A_clean_r1_20261016-115228,1,youtube.com,cloudflare,doh_cold,A,https://cloudflare-dns.com/dns-query,1.792152e+09,1.792152e+09,105.790,ok,NOERROR,4,192.178.173.91;192.178.173.93;192.178.173.190;...,1,NaN,raw_A_clean_r1_20261016-115228.csv
1,A,clean,A_clean_r1_20261016-115228,2,youtube.com,cloudflare,doh_warm,A,https://cloudflare-dns.com/dns-query,1.792152e+09,1.792152e+09,32.964,ok,NOERROR,1,142.251.220.14,1,NaN,raw_A_clean_r1_20261016-115228.csv
2,A,clean,A_clean_r1_20261016-115228,3,youtube.com,cloudflare,classic_udp,A,1.1.1.1,1.792152e+09,1.792152e+09,3.341,ok,NOERROR,4,192.178.174.136;192.178.174.190;192.178.174.91...,1,NaN,raw_A_clean_r1_20261016-115228.csv
3,A,clean,A_clean_r1_20261016-115228,4,google.com,cloudflare,classic_udp,A,1.1.1.1,1.792152e+09,1.792152e+09,5.908,ok,NOERROR,6,192.178.211.101;192.178.211.102;192.178.211.11...,1,NaN,raw_A_clean_r1_20261016-115228.csv
4,A,clean,A_clean_r1_20261016-115228,5,google.com,cloudflare,doh_cold,A,https://cloudflare-dns.com/dns-query,1.792152e+09,1.792152e+09,114.207,ok,NOERROR,6,142.251.106.113;142.251.106.102;142.251.106.10...,1,NaN,raw_A_clean_r1_20261016-115228.csv


## 1. Did the queries work?
`status` should be mostly `ok`. Timeouts/errors are real data - report them, do not delete them.

In [3]:
(df.groupby(["resolver", "method"]).status.value_counts().unstack(fill_value=0))

status                  error  http_505   ok
resolver   method                           
cloudflare classic_udp      0         0  309
           doh_cold         0         0  309
           doh_warm         0         0  309
google     classic_udp      0         0  300
           doh_cold         0         0  300
           doh_warm         0         0  300
quad9      classic_udp      0         0  300
           doh_cold         3       297    0
           doh_warm         3       297    0

## 2. Latency distribution per method and resolver
Boxplots show median, spread and outliers. Outliers (retries, slow resolvers) are *kept* - report mean AND median.

In [ ]:
ok = df[df.status == "ok"]
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=ok, x="resolver", y="rtt_ms", hue="method", ax=ax, showfliers=True, fliersize=2, log_scale=True)
ax.set_ylabel("RTT (ms, log scale)"); ax.set_xlabel("resolver"); ax.set_title("Resolution latency by method")
fig.tight_layout(); fig.savefig(RESULTS / "fig_week1_rtt_boxplot.png", dpi=300); plt.show()


## 3. Numbers for the paper
`summarize` = over all queries; `summarize_by_run` = mean of each run first, then mean ± std across runs (the honest one).

In [5]:
summarize(df).round(2)

,profile,resolver,method,n_total,n_fail,n_ok,mean_ms,median_ms,std_ms,p95_ms,min_ms,max_ms,success_rate
0,clean,cloudflare,classic_udp,309,0,309.0,66.34,5.39,492.78,64.97,2.47,5063.81,1.0
1,clean,cloudflare,doh_cold,309,0,309.0,133.93,116.43,90.21,182.91,98.22,1137.81,1.0
2,clean,cloudflare,doh_warm,309,0,309.0,42.02,34.69,29.84,92.93,28.11,311.56,1.0
3,clean,google,classic_udp,300,0,300.0,56.21,4.30,503.01,15.85,1.90,5117.62,1.0
4,clean,google,doh_cold,300,0,300.0,127.88,114.78,68.79,173.97,97.35,1123.71,1.0
5,clean,google,doh_warm,300,0,300.0,47.05,34.77,30.82,104.21,29.13,277.56,1.0
6,clean,quad9,classic_udp,300,0,300.0,38.90,4.19,409.17,15.82,2.07,5033.00,1.0
7,clean,quad9,doh_cold,300,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
8,clean,quad9,doh_warm,300,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [6]:
summarize_by_run(df).round(2)

,profile,resolver,method,n_runs,mean_of_runs_ms,std_of_runs_ms
0,clean,cloudflare,classic_udp,9,49.98,47.70
1,clean,cloudflare,doh_cold,9,128.27,15.60
2,clean,cloudflare,doh_warm,9,40.22,4.01
3,clean,google,classic_udp,6,56.21,55.80
4,clean,google,doh_cold,6,127.88,7.74
5,clean,google,doh_warm,6,47.05,2.79
6,clean,quad9,classic_udp,6,38.90,51.09


## 4. Do the two locations differ?
Different ISPs/paths between teammates is itself a useful robustness signal.

In [7]:
summarize(df, by=["person", "resolver", "method"])[["person","resolver","method","n_ok","mean_ms","median_ms","p95_ms"]].round(2)

,person,resolver,method,n_ok,mean_ms,median_ms,p95_ms
0,A,cloudflare,classic_udp,153.0,59.62,28.18,79.34
1,A,cloudflare,doh_cold,153.0,137.97,118.00,194.50
2,A,cloudflare,doh_warm,153.0,43.29,35.85,95.40
3,A,google,classic_udp,150.0,39.61,4.00,14.70
4,A,google,doh_cold,150.0,126.15,116.36,179.26
5,A,google,doh_warm,150.0,48.00,35.31,106.55
6,A,quad9,classic_udp,150.0,71.78,3.94,14.88
7,A,quad9,doh_cold,NaN,NaN,NaN,NaN
8,A,quad9,doh_warm,NaN,NaN,NaN,NaN
9,B,cloudflare,classic_udp,156.0,72.94,4.43,36.44


## 5. Privacy: what can a passive observer read?
The latency numbers above are only half of the trade-off. `capture/privacy_audit.py` searches your pcap for the domains you queried: classic DNS should expose ~100 %, DoH ~0 %.

```
python capture/privacy_audit.py capture/pcaps/<file>.pcap --client "results/raw_A_clean_r1_*.csv" --out results/privacy_audit_A.csv
```

In [8]:
files = sorted(RESULTS.glob("privacy_audit_*.csv"))
if files:
    pa = pd.concat([pd.read_csv(f).assign(file=f.name) for f in files])
    display(pa[["file", "channel", "domains_queried", "domains_visible_to_observer", "exposure_rate", "sni_seen"]].round(3))
else:
    print("no privacy_audit_*.csv yet - run capture/privacy_audit.py (see above)")

no privacy_audit_*.csv yet - run capture/privacy_audit.py (see above)
